### Sample generation for annual probability rasters
1. Create a binary mask of all the agricultural lands from 2000-2022 (note that the land cover dataset for 2012 is not available).
2. Identify the stable agricultural areas by finding the intersection of all the binary masks.
3. Remove the small pixel clusters, field boundaries using the morphological operations.
4. Convert the final stable agricultural land pixels to points
5. Remove all the points except one inside a 1000/2000 meter buffer ensuring all the points are far from each other (spatial autocorrelation).
6. Final check using a phenology profile of the points to ensure they strictly exhibit agricultural characteristics (sharp increase and decrease in NDVI).
7. Repeat the same for the stable non-agricultural lands. Also I will do a stratified sampling using the 2022 land cover dataset so that the model gets trained will all non-cropland classses and does not get confused between different classes. Importantly, I will ensure that the dominant land cover classes like forest and rangeland, bare soil/sand, water and builtup are represented in the training samples.

In [1]:
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
Map = geemap.Map()
from helpers import config

loaded config!


# First for stable Ag

### Steps 1-2: Stable ag mask generation

In [2]:
config.ROI = ee.FeatureCollection("projects/ee-joshisur231/assets/nepal_province").geometry()
dem = ee.Image("USGS/SRTMGL1_003")
geo_region = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal").rename("geoReg")

slope_mask = ee.Terrain.slope(dem).lte(35)
ag_2003 = ee.ImageCollection("users/potapovpeter/Global_cropland_2003").first()
ag_2007 = ee.ImageCollection("users/potapovpeter/Global_cropland_2007").first()
ag_2011 = ee.ImageCollection("users/potapovpeter/Global_cropland_2011").first()
ag_2015 = ee.ImageCollection("users/potapovpeter/Global_cropland_2015").first()
ag_2019 = ee.ImageCollection("users/potapovpeter/Global_cropland_2019").first()

ag_stable_p = ag_2003.add(ag_2007).add(ag_2011).add(ag_2015).add(ag_2019).eq(5)
frtc_lc_ic = ee.ImageCollection("projects/ee-joshisur231/assets/landcover_frtc_2000-2022_nepal")
# china_lc_ic = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual").mosaic()

frtc_ag_ic = frtc_lc_ic.map(lambda image: image.eq(7).rename("ag"))
# china_ag_ic = ee.ImageCollection.fromImages(china_lc_ic.bandNames().map(lambda band_name: china_lc_ic.select([band_name]).eq(10).Or(china_lc_ic.select([band_name]).eq(11)).Or(china_lc_ic.select([band_name]).eq(20)).rename("ag")))

frtc_stable_ag_mask = frtc_ag_ic.sum().eq(22)

# china_stable_ag_mask = china_ag_ic.sum().eq(23)

# stable_ag_mask = frtc_stable_ag_mask.add(china_stable_ag_mask).eq(2)
# stable_ag_mask = frtc_stable_ag_mask.updateMask(slope_mask) #Using Frtc only because sample of combined approach did not represent hilly areas adequately
# stable_ag_mask = china_stable_ag_mask.updateMask(slope_mask)
stable_ag_mask = frtc_stable_ag_mask.add(ag_stable_p).eq(2).updateMask(slope_mask)

### Steps 3: Stable ag mask morphological op

In [3]:
stable_ag_eroded_mask = stable_ag_mask.focalMin(radius=2, units='pixels') #Removing ag field boundary by 2 pixel
patch_size = stable_ag_eroded_mask.connectedPixelCount(maxSize=10, eightConnected=True)
stable_ag_final_mask = stable_ag_eroded_mask.updateMask(patch_size.gte(10))

In [ ]:
# geemap.ee_export_image_to_asset(
#     assetId = "projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_ag_points_glad",
#     image= stable_ag_final_mask,
#     scale=30,
#     region = config.ROI,
#     maxPixels = 1e13,
# )

In [ ]:
# geo_region = ee.Image("projects/ee-joshisur231/assets/pa_effectiveness/geoReg_nepal").rename("geoReg")
# geo_stable_crop = stable_ag_final_mask.addBands(geo_region)
# area_stable_ag_final_mask = geo_stable_crop.reduceRegion(
#     reducer = ee.Reducer.sum().unweighted().group(groupField = 1, groupName = "geo"),
#     geometry = config.ROI,
#     scale = 30,
#     maxPixels = 474870364
# )

In [4]:
stable_ag_mask.reduceRegion(geometry = config.ROI, reducer=ee.Reducer.sum().unweighted(), scale=config.SCALE, maxPixels=287435182)

KeyboardInterrupt: 

In [ ]:
stable_ag_eroded_mask.reduceRegion(geometry = config.ROI, reducer=ee.Reducer.sum().unweighted(), scale=config.SCALE, maxPixels=287435182)

In [4]:
stable_ag_final_mask.reduceRegion(geometry = config.ROI, reducer=ee.Reducer.sum().unweighted(), scale=config.SCALE, maxPixels=287435182)

In [5]:
Map.addLayer(ee.Terrain.slope(dem), {"min":5, "max":50}, "slope")
Map.addLayer(stable_ag_mask.selfMask(), {"palette":["red"]}, "stable_ag")
# Map.addLayer(stable_ag_eroded_mask.selfMask(), {"palette":["green"]}, "stable_ag_eroded")
Map.addLayer(stable_ag_final_mask.selfMask(), {"palette":["blue"]}, "stable_ag_final")
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

### Step 4-5: Stable ag points generation and thining

In [3]:
stable_ag_points = ee.Image("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_ag_points_glad").selfMask().rename('ag').stratifiedSample(
    numPoints=10000000,     
    classBand='ag', 
    region=config.ROI, 
    scale=30, 
    geometries=True,
    dropNulls=True,
    tileScale = 16
)

In [4]:
def apply_spatial_thinning(points, distance_meters):
    """
    Removes spatial autocorrelation by ensuring no two points 
    are within the specified distance of each other.
    """
    # 1. Add a random value to each point
    points_with_random = points.randomColumn('random')

    # 2. Define a spatial filter for the 1,000-meter radius
    dist_filter = ee.Filter.withinDistance(
        distance=distance_meters,
        leftField='.geo',
        rightField='.geo',
        maxError=10
    )

    # 3. Create a join to find all neighbors within that distance
    join = ee.Join.saveAll(
        matchesKey='neighbors',
        measureKey='distance'
    )

    # 4. Apply the join to the FeatureCollection
    joined_points = join.apply(points_with_random, points_with_random, dist_filter)

    # 5. Function to evaluate each neighborhood
    def check_if_max(feature):
        # Get the list of all points within 1,000m (including itself)
        neighbors = ee.List(feature.get('neighbors'))
        
        # Extract the random values of all these neighbors
        neighbor_randoms = neighbors.map(lambda f: ee.Feature(f).get('random'))
        
        # Find the maximum random value in this cluster
        max_random = neighbor_randoms.reduce(ee.Reducer.max())
        
        # If THIS point's random value is the maximum, mark it to be kept
        is_max = ee.Number(feature.get('random')).eq(max_random)
        
        return feature.set('keep', is_max)

    # 6. Apply the evaluation and filter out the losers
    thinned_points = joined_points.map(check_if_max).filter(ee.Filter.eq('keep', 1))

    # 7. Clean up the temporary properties we added so your data stays clean
    def cleanup(f):
        return f.set('keep', None).set('neighbors', None).set('random', None)
        
    return thinned_points.map(cleanup)

In [5]:
stable_ag_points_filtered = apply_spatial_thinning(stable_ag_points, 2000)

In [ ]:
stable_ag_points_filtered.size()

In [ ]:
# geemap.ee_export_vector_to_drive(
#     collection=ee.FeatureCollection(ee.Feature(None, ee.Dictionary({"a":(stable_ag_points_filtered.size())}))),
#     description="size",
#     fileFormat = "CSV"
# )

Exporting size... Please check the Task Manager from the JavaScript Code Editor.


In [6]:
geemap.ee_export_vector_to_asset(
    collection=stable_ag_points_filtered,
    description = "stable_ag_points_filtered",
    assetId = "projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_ag_points_filtered_glad2"
)

projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_ag_points_filtered_glad2
Exporting stable_ag_points_filtered... Please check the Task Manager from the JavaScript Code Editor.


In [ ]:
geemap.ee_export_vector_to_drive(
  collection= table, 
  description= "original_geometries_ag_points", 
  fileFormat= "CSV")

In [56]:
# Map.addLayer(stable_ag_final_mask, {"palette":["white", "red"]}, "stable_ag")
# Map.addLayer(stable_ag_points_filtered, {}, "stable_ag_points_filtered")
# Map

### Extract Landsat data and Export to drive

In [11]:
start_date = "2000-01-01"
end_date = "2022-12-31"
# stable_ag_samples = stable_ag_points_filtered.map(lambda feat: feat.set("point_id", feat.id())) #ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/stable_ag_points_filtered")\

test_points = ee.FeatureCollection(
        [ee.Feature(
            ee.Geometry.Point([82.16041111263897, 29.36141014503806]),
            {
              "lc": "rangeland",
              "system:index": "0",
              "point_id": 1
            }),
        ee.Feature(
            ee.Geometry.Point([82.1582009724107, 29.358623611626218]),
            {
              "lc": "rangeland",
              "system:index": "1",
              "point_id": 2
            }),
        ee.Feature(
            ee.Geometry.Point([82.1590485504594, 29.358221521652027]),
            {
              "lc": "rangeland",
              "system:index": "2",
              "point_id": 3
            }),
        ee.Feature(
            ee.Geometry.Point([82.39206584787914, 29.366226599664596]),
            {
              "lc": "rangeland",
              "system:index": "3",
              "point_id": 4
            }),
        ee.Feature(
            ee.Geometry.Point([82.39461931086132, 29.366609957679856]),
            {
              "lc": "rangeland",
              "system:index": "4",
              "point_id": 5
            }),
        ee.Feature(
            ee.Geometry.Point([82.39413919544765, 29.359931374118506]),
            {
              "lc": "rangeland",
              "system:index": "5",
              "point_id": 6
            }),
        ee.Feature(
            ee.Geometry.Point([82.38909781227366, 29.277604977340843]),
            {
              "lc": "rangeland",
              "system:index": "6",
              "point_id": 7
            }),
        ee.Feature(
            ee.Geometry.Point([82.40008414039866, 29.271559311026945]),
            {
              "lc": "rangeland",
              "system:index": "7",
              "point_id": 8
            }),
        ee.Feature(
            ee.Geometry.Point([82.40186512718455, 29.269818542038376]),
            {
              "lc": "rangeland",
              "system:index": "8",
              "point_id": 9
            }),
        ee.Feature(
            ee.Geometry.Point([82.51337691583186, 29.168292061322934]),
            {
              "lc": "rangeland",
              "system:index": "9",
              "point_id": 10
            }),
        ee.Feature(
            ee.Geometry.Point([82.520608151336, 29.169752339850977]),
            {
              "lc": "rangeland",
              "system:index": "10",
              "point_id": 11        
            }),
        ee.Feature(
            ee.Geometry.Point([82.0415644518115,29.3968734045009]),
            {
              "lc": "crop",
              "system:index": "11",
              "point_id": 12        
            }),
        ee.Feature(
            ee.Geometry.Point([82.0553074012582,29.4213984602949]),
            {
              "lc": "crop",
              "system:index": "12",
              "point_id": 13       
            }),
        ee.Feature(
            ee.Geometry.Point([81.9809251365857,29.2467666857387]),
            {
              "lc": "crop",
              "system:index": "13",
              "point_id": 14        
            }),
        ee.Feature(
            ee.Geometry.Point([86.70812964087,27.4583996172438]),
            {
              "lc": "crop",
              "system:index": "14",
              "point_id": 15        
            }),
        ee.Feature(
            ee.Geometry.Point([82.7363191505446,29.0015250460013]),
            {
              "lc": "crop",
              "system:index": "15",
              "point_id": 16        
            }),
        ee.Feature(
            ee.Geometry.Point([80.4334877571094,29.7315869068757]),
            {
              "lc": "crop",
              "system:index": "16",
              "point_id": 17       
            }),
        ee.Feature(
            ee.Geometry.Point([81.894686122212,29.7911426605455]),
            {
              "lc": "crop",
              "system:index": "17",
              "point_id": 18        
            }),
        ee.Feature(
            ee.Geometry.Point([85.7638213516807,27.736246203884]),
            {
              "lc": "crop",
              "system:index": "18",
              "point_id": 19        
            }),
        ee.Feature(
            ee.Geometry.Point([81.3096163911915,29.4181656120311]),
            {
              "lc": "crop",
              "system:index": "19",
              "point_id": 20       
            }),
        ee.Feature(
            ee.Geometry.Point([82.1434326153774,29.4995531244586]),
            {
              "lc": "crop",
              "system:index": "20",
              "point_id": 21       
            }),
        ee.Feature(
            ee.Geometry.Point([86.226814273161,27.8435054206235]),
            {
              "lc": "crop",
              "system:index": "21",
              "point_id": 22        
            })
            ])

In [1]:
import ee
# ee.Authenticate()
ee.Initialize(project="ee-joshisur231")
import geemap
Map = geemap.Map()
# from helpers import config

In [2]:
start_date = "2000-01-01"
end_date = "2022-12-31"
roi = ee.FeatureCollection("projects/ee-joshisur231/assets/pa_effectiveness/nepal_boundary")
points = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/crop_stable_glad2000m_arcgis")

In [3]:
l8_ndvi_col = ee.ImageCollection("LANDSAT/COMPOSITES/C02/T1_L2_8DAY_NDVI")\
    .filterBounds(roi)\
    .filterDate(start_date, end_date)

In [5]:
def extract_point_values(image):
    l_with_time_band = image.addBands(image.metadata("system:time_start").rename("time"))
    points_with_ndvi = l_with_time_band.reduceRegions(collection = points, scale=30, reducer=ee.Reducer.first())
    return points_with_ndvi

points_with_ndvi = l8_ndvi_col.map(extract_point_values).flatten()#.map(lambda feat: feat.set("coords", feat.geometry().coordinates()))

In [6]:
geemap.ee_export_vector_to_drive(
    collection=points_with_ndvi,
    description="stable_crop_withNDVI_glad_arcgis",
    fileFormat = "CSV"
)

Exporting stable_crop_withNDVI_glad_arcgis... Please check the Task Manager from the JavaScript Code Editor.
